# Notebook 04: Different Types of Regression
**Lab Assignment:** Assignment 4  
**Project Role:** Multiple regression, Ridge L2, Lasso L1, and Logistic click prediction modeling.

### Mathematical Formulations
- **Multiple Regression (Normal Equation):** $\hat{oldsymbol{eta}} = (X^T X)^{-1} X^T \mathbf{y}$
- **Multiple Regression (Gradient Descent):** $oldsymbol{eta}^{(t+1)} = oldsymbol{eta}^{(t)} - rac{lpha}{n} X^T (X oldsymbol{eta}^{(t)} - \mathbf{y})$
- **Ridge Regression (Closed-Form):** $\hat{oldsymbol{eta}}_{	ext{ridge}} = (X^T X + \lambda I')^{-1} X^T \mathbf{y}$
- **Lasso Regression (Coordinate Descent):** $eta_j \leftarrow rac{S(ho_j, \lambda)}{\|x_j\|^2}$ where $S(z, \gamma) = 	ext{sign}(z)\max(|z| - \gamma, 0)$
- **Logistic Regression:** $P(y=1|\mathbf{x}) = \sigma(\mathbf{w}^T \mathbf{x} + b) = rac{1}{1 + e^{-(\mathbf{w}^T \mathbf{x} + b)}}$


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
from src.data_loader import load_data
from src.preprocessing import clean_dataframes, temporal_train_val_test_split
from src.features import SubstackFeaturePipeline
from src.regression import MultipleLinearRegressionScratch, RidgeRegressionScratch, LassoRegressionScratch, LogisticRegressionScratch, r2_score_scratch, root_mean_squared_error_scratch, mean_absolute_error_scratch
df_posts, df_interactions = load_data()
df_posts, df_interactions = clean_dataframes(df_posts, df_interactions)
train_inter, val_inter, test_inter = temporal_train_val_test_split(df_interactions)
pipeline = SubstackFeaturePipeline().fit(df_posts, train_inter)
train_feat = pipeline.extract_features(df_posts, train_inter)
test_feat = pipeline.extract_features(df_posts, test_inter)
feat_cols = ['word_count', 'title_length', 'readability_score', 'recency_decay', 'item_popularity', 'user_avg_read_time', 'topic_affinity_score']
X_train = train_feat[feat_cols].values
y_train_rt = train_feat['read_time'].values
y_train_click = train_feat['clicked'].values
X_test = test_feat[feat_cols].values
y_test_rt = test_feat['read_time'].values
y_test_click = test_feat['clicked'].values


In [ ]:
mlr_norm = MultipleLinearRegressionScratch().fit_normal_equation(X_train, y_train_rt)
mlr_gd = MultipleLinearRegressionScratch().fit_gradient_descent(X_train, y_train_rt, learning_rate=0.01, epochs=500)
sk_linear = LinearRegression().fit(X_train, y_train_rt)
ridge_scratch = RidgeRegressionScratch(alpha=1.0).fit(X_train, y_train_rt)
sk_ridge = Ridge(alpha=1.0).fit(X_train, y_train_rt)
lasso_scratch = LassoRegressionScratch(alpha=0.05, max_iter=300).fit(X_train, y_train_rt)
sk_lasso = Lasso(alpha=0.05).fit(X_train, y_train_rt)
logistic_scratch = LogisticRegressionScratch(learning_rate=0.05, epochs=300).fit(X_train, y_train_click)
sk_logistic = LogisticRegression().fit(X_train, y_train_click)


In [ ]:
results = [{'Model': 'MLR (Normal Eq - Scratch)', 'R2': r2_score_scratch(y_test_rt, mlr_norm.predict(X_test)), 'RMSE': root_mean_squared_error_scratch(y_test_rt, mlr_norm.predict(X_test)), 'MAE': mean_absolute_error_scratch(y_test_rt, mlr_norm.predict(X_test))}, {'Model': 'MLR (Sklearn)', 'R2': sk_linear.score(X_test, y_test_rt), 'RMSE': np.sqrt(np.mean((y_test_rt - sk_linear.predict(X_test)) ** 2)), 'MAE': np.mean(np.abs(y_test_rt - sk_linear.predict(X_test)))}, {'Model': 'Ridge (Scratch)', 'R2': r2_score_scratch(y_test_rt, ridge_scratch.predict(X_test)), 'RMSE': root_mean_squared_error_scratch(y_test_rt, ridge_scratch.predict(X_test)), 'MAE': mean_absolute_error_scratch(y_test_rt, ridge_scratch.predict(X_test))}, {'Model': 'Ridge (Sklearn)', 'R2': sk_ridge.score(X_test, y_test_rt), 'RMSE': np.sqrt(np.mean((y_test_rt - sk_ridge.predict(X_test)) ** 2)), 'MAE': np.mean(np.abs(y_test_rt - sk_ridge.predict(X_test)))}, {'Model': 'Lasso (Scratch)', 'R2': r2_score_scratch(y_test_rt, lasso_scratch.predict(X_test)), 'RMSE': root_mean_squared_error_scratch(y_test_rt, lasso_scratch.predict(X_test)), 'MAE': mean_absolute_error_scratch(y_test_rt, lasso_scratch.predict(X_test))}]
print(pd.DataFrame(results).to_string(index=False))
log_preds_scratch = logistic_scratch.predict(X_test)
log_preds_sk = sk_logistic.predict(X_test)
print('\nLogistic Regression (Scratch) Accuracy:', accuracy_score(y_test_click, log_preds_scratch))
print('Logistic Regression (Sklearn) Accuracy:', accuracy_score(y_test_click, log_preds_sk))


### Interpretation & Findings
1. **Regularization Benefits:** Ridge regression attenuates feature collinearity between word count and reading ease, yielding lower test variance.
2. **Sparsity in Lasso:** Coordinate descent successfully drives zero coefficients for non-informative features, acting as an embedded feature selector.
3. **Probability Calibration:** The from-scratch logistic sigmoid produces calibrated probabilities that serve as confidence scores in downstream hybrid ranking.

### Limitations
- Linear decision surfaces cannot capture complex non-linear feature interactions without explicit polynomial feature expansion.
